# Hebrew Letterform Classifier — generate data on Kaggle + train (Phase 1 + Phase 4)

Downloads `fonts.zip` from Google Drive, renders the synthetic glyph dataset inside
Kaggle, then trains the 27-class CNN.

**Setup:**
1. Upload `fonts.zip` to Google Drive, share it **"Anyone with the link"**, and paste its
   link into `ZIP_URL` in the first code cell.
2. **Settings → Internet → ON**.
3. **Settings → Accelerator → GPU**.
4. **Run All**. Outputs land in `/kaggle/working`.

In [ ]:
# ── 0. download fonts.zip from Google Drive (single file) ───────────────
!pip install -q -U gdown
import gdown, glob, os, re, zipfile

# Share link of YOUR fonts.zip (Share > Anyone with the link):
ZIP_URL = 'https://drive.google.com/file/d/1DnF8UGjGQlZCUPx2F02gIg4nNiqZF5Xh/view?usp=sharing'

# Extract the file id (version-proof; avoids gdown's fuzzy/folder bugs).
m = re.search(r'/d/([\w-]+)|[?&]id=([\w-]+)', ZIP_URL)
FILE_ID = next(g for g in m.groups() if g)
gdown.download(f'https://drive.google.com/uc?id={FILE_ID}', '/kaggle/working/fonts.zip', quiet=False)
with zipfile.ZipFile('/kaggle/working/fonts.zip') as z:
    z.extractall('/kaggle/working/fonts')

FONTS = sorted(glob.glob('/kaggle/working/fonts/**/*.ttf', recursive=True) +
               glob.glob('/kaggle/working/fonts/**/*.otf', recursive=True))
assert FONTS, 'No fonts extracted — check ZIP_URL and that the file is shared.'
print(f'extracted {len(FONTS)} fonts, e.g.:'); print(' ', FONTS[0])


In [ ]:
# ── 1. params ──────────────────────────────────────────────────────────
OUT = '/kaggle/working'
ALPHABET = list('אבגדהוזחטיכלמנסעפצקרשת') + list('ךםןףץ')   # 22 base + 5 final = 27
CLS = {c:i for i,c in enumerate(ALPHABET)}
RENDER_PX   = 128     # nominal glyph render height
OUT_SIZE    = 64      # final square tile fed to the CNN
PER_CLASS   = 30      # augmented samples per letter per font (40*27*30 ~ 32k)
PAD_RATIO   = 0.18
SEED        = 1234
EPOCHS, BATCH, LR, VAL_SPLIT = 25, 128, 3e-4, 0.1
print('classes:', len(ALPHABET))


In [ ]:
# ── 2. rendering + augmentation (Phase-1 logic, inlined) ────────────────
import numpy as np, cv2
from PIL import Image, ImageDraw, ImageFont
rng = np.random.default_rng(SEED)

def render_glyph(font, ch):
    asc, desc = font.getmetrics()
    box = font.getbbox(ch)
    if box is None: return None
    w = max(box[2]-box[0],1)+8; h = asc+desc+8
    img = Image.new('L',(w,h),0); ImageDraw.Draw(img).text((4-box[0],4),ch,font=font,fill=255)
    a = np.asarray(img); ys,xs = np.where(a>10)
    if len(xs)==0: return None
    return a[ys.min():ys.max()+1, xs.min():xs.max()+1]

def augment(ink):
    h,w = ink.shape
    # elastic
    alpha = rng.uniform(20,45); sigma = rng.uniform(4,7)
    dx = cv2.GaussianBlur(rng.random((h,w)).astype(np.float32)*2-1,(0,0),sigma)*alpha
    dy = cv2.GaussianBlur(rng.random((h,w)).astype(np.float32)*2-1,(0,0),sigma)*alpha
    gx,gy = np.meshgrid(np.arange(w),np.arange(h))
    ink = cv2.remap(ink,(gx+dx).astype(np.float32),(gy+dy).astype(np.float32),cv2.INTER_LINEAR,borderValue=0)
    # rotation + slant
    R = cv2.getRotationMatrix2D((w/2,h/2), rng.uniform(-8,8), 1.0)
    ink = cv2.warpAffine(ink,R,(w,h),borderValue=0)
    M = np.float32([[1, rng.uniform(-1,1)*0.18, 0],[0,1,0]])
    ink = cv2.warpAffine(ink,M,(w,h),borderValue=0)
    # stroke thicken/thin
    op = rng.choice(['erode','dilate','none'])
    if op!='none':
        k = int(rng.integers(1,4)); ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(k,k))
        ink = (cv2.erode if op=='erode' else cv2.dilate)(ink,ker)
    # ink bleed
    if rng.random()<0.3:
        ink = cv2.dilate(ink,np.ones((2,2),np.float32)); ink = cv2.GaussianBlur(ink,(0,0),0.6)
    ink = cv2.GaussianBlur(ink,(0,0),rng.uniform(0.01,1.2))
    return np.clip(ink,0,1)

def compose(ink):  # -> black-ink-on-white uint8 (like a scanned crop)
    h,w = ink.shape
    if rng.random()<0.5:
        bg = rng.uniform(0.85,1.0); tex = cv2.GaussianBlur(rng.random((h,w)).astype(np.float32),(0,0),3)
        paper = np.clip(bg-0.06*(tex-tex.mean()),0,1)
    else:
        paper = np.ones((h,w),np.float32)
    out = paper*(1-ink) + 0.05*ink
    out = out + rng.normal(0, rng.uniform(0,12)/255.0, out.shape)
    return (np.clip(out,0,1)*255).astype(np.uint8)

def square(img, size=OUT_SIZE, pad=PAD_RATIO, bg=255):
    h,w = img.shape; p = int(max(h,w)*pad); side = max(h,w)+2*p
    c = np.full((side,side),bg,np.uint8); c[(side-h)//2:(side-h)//2+h,(side-w)//2:(side-w)//2+w]=img
    return cv2.resize(c,(size,size),interpolation=cv2.INTER_AREA)


In [ ]:
# ── 3. generate the dataset in memory ──────────────────────────────────
from tqdm.auto import tqdm
X, y = [], []
for fp in tqdm(FONTS, desc='fonts'):
    try: font = ImageFont.truetype(fp, RENDER_PX)
    except Exception: continue
    for ch in ALPHABET:
        base = render_glyph(font, ch)
        if base is None: continue
        ink0 = base.astype(np.float32)/255.0
        for _ in range(PER_CLASS):
            tile = square(compose(augment(ink0.copy())))
            X.append(tile); y.append(CLS[ch])
X = np.stack(X); y = np.asarray(y, np.int64)
print('generated glyphs:', len(X), '| classes present:', len(set(y.tolist())))


In [ ]:
# ── 4. peek at a few samples ───────────────────────────────────────────
import matplotlib.pyplot as plt
idx = rng.integers(0, len(X), 10)
fig,ax = plt.subplots(1,10,figsize=(15,2))
for a,i in zip(ax, idx): a.imshow(X[i],cmap='gray'); a.set_title(ALPHABET[y[i]]); a.axis('off')
plt.show()


In [ ]:
# ── 5. model ───────────────────────────────────────────────────────────
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device:', device)
def blk(ci,co): return nn.Sequential(nn.Conv2d(ci,co,3,padding=1),nn.BatchNorm2d(co,momentum=0.05),nn.ReLU(True),
                                     nn.Conv2d(co,co,3,padding=1),nn.BatchNorm2d(co,momentum=0.05),nn.ReLU(True),nn.MaxPool2d(2))
num_classes = len(ALPHABET)
model = nn.Sequential(blk(1,32),blk(32,64),blk(64,128),nn.AdaptiveAvgPool2d(1),nn.Flatten(),
                      nn.Linear(128,256),nn.ReLU(True),nn.Dropout(0.4),nn.Linear(256,num_classes)).to(device)


In [ ]:
# ── 6. train ───────────────────────────────────────────────────────────
Xt = torch.from_numpy(X).float().div_(255).unsqueeze(1); yt = torch.from_numpy(y)
full = TensorDataset(Xt, yt); n_val = int(len(full)*VAL_SPLIT)
g = torch.Generator().manual_seed(SEED)
tr,va = random_split(full,[len(full)-n_val,n_val],generator=g)
tr_dl = DataLoader(tr,batch_size=BATCH,shuffle=True,num_workers=2)
va_dl = DataLoader(va,batch_size=BATCH,num_workers=2)
opt = torch.optim.Adam(model.parameters(),lr=LR)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt,EPOCHS); crit = nn.CrossEntropyLoss(label_smoothing=0.05)
best_acc, best_cm = 0.0, None
for ep in range(EPOCHS):
    model.train(); tot=0.0
    for xb,yb in tr_dl:
        xb,yb=xb.to(device),yb.to(device); opt.zero_grad()
        loss=crit(model(xb),yb); loss.backward(); opt.step(); tot+=loss.item()*len(xb)
    sched.step(); model.eval(); correct=0; cm=np.zeros((num_classes,num_classes),np.int64)
    with torch.no_grad():
        for xb,yb in va_dl:
            p=model(xb.to(device)).argmax(1).cpu(); correct+=(p==yb).sum().item()
            for t,q in zip(yb.numpy(),p.numpy()): cm[t,q]+=1
    acc=correct/max(len(va),1)
    print(f'epoch {ep+1:2d}/{EPOCHS}  loss={tot/len(tr):.4f}  val_acc={acc:.4f}')
    if acc>=best_acc:
        best_acc,best_cm=acc,cm
        torch.save({'state_dict':model.state_dict(),'classes':ALPHABET,'img':OUT_SIZE}, f'{OUT}/classifier.pth')
print('BEST val_acc =', best_acc)


In [ ]:
# ── 7. confusion matrix + save metrics ─────────────────────────────────
import json
json.dump({'val_acc':best_acc,'classes':ALPHABET}, open(f'{OUT}/metrics.json','w'), ensure_ascii=False, indent=2)
cmn = best_cm/best_cm.sum(1,keepdims=True).clip(min=1)
fig,ax=plt.subplots(figsize=(10,9)); ax.imshow(cmn,cmap='viridis')
ax.set_xticks(range(num_classes)); ax.set_xticklabels(ALPHABET)
ax.set_yticks(range(num_classes)); ax.set_yticklabels(ALPHABET)
ax.set_xlabel('predicted'); ax.set_ylabel('true'); ax.set_title('Confusion (row-normalized)')
fig.tight_layout(); fig.savefig(f'{OUT}/confusion_matrix.png',dpi=120); plt.show()
print('outputs:', os.listdir(OUT))
